In [ ]:
from Excited_VMC import ha,hi,SingleStateAnsatz,create_machine,ground_model,excited_model
import jax
import jax.numpy as jnp
import netket as nk
import netket.experimental as nkx
import numpy as np
from pyscf import gto, scf, fci
from flax import linen as nn
import flax.nnx as nnx
import optax
from tqdm import tqdm
from functools import partial
from jax import flatten_util
import orbax.checkpoint as ocp
from pathlib import Path
from Excited_VMC import exact_energy_efficient,create_machine,sampler
from Excited_VMC import compute_local_energies,statistics,compute_qgt,E_fcis,forces_expect_with_penalty,forces_expect_with_multiple_penalty

#ground_model = SingleStateAnsatz(4,12, rngs=nnx.Rngs(21))
excited1_model = SingleStateAnsatz(4,12, rngs=nnx.Rngs(22))
excited2_model = SingleStateAnsatz(4,12, rngs=nnx.Rngs(23))

sampler_state = sampler.init_state(machine_es2,params_es2,seed=12)
samples,sampler_state = sampler.sample(machine_es2,params_es2,chain_length=20)


In [9]:
import orbax.checkpoint as ocp
from pathlib import Path

# 1. 获取当前工作目录的绝对路径
current_dir = Path.cwd().parent  # 或者用 Path(__file__).parent 如果在脚本中
ckpt_dir = current_dir / "暂存态"
ckpt_dir.mkdir(exist_ok=True)

# 2. 使用 PyTreeCheckpointer 而不是 StandardCheckpointer
checkpointer = ocp.PyTreeCheckpointer()

# 3. 保存模型（使用绝对路径）
# _, state = nnx.split(model)
save_path = str(ckpt_dir / "excited_state1")  # 转为字符串
excited1_model = checkpointer.restore(save_path)
save_path = str(ckpt_dir / "ground_state")  # 转为字符串
ground_model = checkpointer.restore(save_path)

machine_gs,graphdef,params_gs = create_machine(ground_model)
machine_es1,graphdef,params_es1 = create_machine(excited1_model)
machine_es2,graphdef,params_es2 = create_machine(excited2_model)


/opt/miniconda3/envs/Neural/lib/python3.11/site-packages/orbax/checkpoint/_src/serialization/type_handlers.py:1256: UserWarning: Sharding info not provided when restoring. Populating sharding info from sharding file. Please note restoration time will be slightly increased due to reading from file. Note also that this option is unsafe when restoring on a different topology than the checkpoint was saved with.
  warnings.warn(


In [10]:
total_loss, E_mean, E_std, total_grad, overlap_sq = forces_expect_with_multiple_penalty(
machine_list=[machine_gs,machine_es1],
params_list=[params_gs,params_es1],
target_machine=machine_es2,
target_params=params_es2,
sigma=samples.reshape(-1,4),
lam_list=[1.0,0.8,0.8]
)
total_loss, E_mean, E_std, overlap_sq 

TypeError: conjugate requires ndarray or scalar arguments, got <class 'flax.nnx.statelib.State'> at position 0.

In [ ]:
# ===================== 6. 初始化 =====================
from Excited_VMC import exact_energy_efficient
rngs = nnx.Rngs(21)
sampler_state = sampler.init_state(machine_es, params_es, seed=1)

optimizer = optax.sgd(learning_rate=0.01)
opt_state = optimizer.init(params_es)

N_ITER = 400
N_SAMPLES = 1008

history = {
    'step': [],
    'energy': [],
    'energy_std': [],
    'overlap_sq': [],
    'error': []
}

# ===================== 7. 训练循环 =====================
print("\n" + "="*60)
print("开始 激发态 VMC 训练 (能量惩罚 + 正交约束 + 自然梯度)")
print("目标：第一激发态 E1")
print("="*60)

for step in range(N_ITER):
    # 1. 采样

    sampler_state = sampler.reset(machine_es, params_es, sampler_state)
    samples, sampler_state = sampler.sample(
        machine_es, params_es, state=sampler_state,
        chain_length=20
    )
    samples = samples.reshape(-1, hi.size)
    # 2. ✅ 正确调用带惩罚的梯度（关键修复）
    total_loss, E_mean, E_std, total_grad, overlap_sq = forces_expect_with_penalty(
    machine_gs,machine_es,
    params_gs,params_es,               
    samples.reshape(-1,4),
    lam=1.0     # 正交惩罚强度
    )

    # 3. ✅ 计算 QGT 自然梯度
    qgt_reg, unravel_fn = compute_qgt(machine_es, params_es, samples, diag_shift=0.001)
    grad_flat, unravel_grad_fn = flatten_util.ravel_pytree(total_grad)
    
    natural_grad_flat = jnp.linalg.solve(qgt_reg.astype(complex), grad_flat)
    natural_grad = unravel_grad_fn(natural_grad_flat)
    
    # 4. ✅ 用自然梯度更新参数
    updates, opt_state = optimizer.update(natural_grad, opt_state, params_es)
    params_es = optax.apply_updates(params_es, updates)

    # 5. 日志输出
    if step % 50 == 0 or step == N_ITER - 1:
        error = jnp.abs(E_mean.real - E_fcis[1])  # 激发态！和 E1 比，不是 E0！
        history['step'].append(step)
        history['energy'].append(float(E_mean.real))
        history['energy_std'].append(float(E_std))
        history['overlap_sq'].append(float(overlap_sq))
        history['error'].append(float(error))
        
        print(f"Step {step:3d} | "
              f"E: {E_mean.real:.8f} ± {E_std:.6f} | "
              f"FCI-E1: {E_fcis[1]:.8f} | "
              f"Overlap²: {overlap_sq:.6f} | "
              f"Err: {error:.6f}")

# ===================== 最终结果 =====================
# final_loss, final_energy, final_std, final_grad, final_overlap = forces_expect_with_penalty(
#     machine_es,
#     params_gs,params_es,
#     samples.reshape(-1,4), 1.0
# )
final_energy = history['energy'][-1]
final_error = history['error'][-1]

print("\n" + "="*60)
print(f"训练完成！")
print(f"第一激发态能量：{final_energy:.8f}Ha")
print(f"FCI 激发态基准：{E_fcis[1]:.8f} Ha")
# print(f"正交重叠 |⟨Ψ0|Ψ1⟩|² = {final_overlap:.8f} (越小越好)")
print(f"绝对误差：{final_error:.6f} Ha")
print("="*60)

In [ ]:
import orbax.checkpoint as ocp
from pathlib import Path

# 1. 获取当前工作目录的绝对路径
current_dir = Path.cwd().parent  # 或者用 Path(__file__).parent 如果在脚本中
ckpt_dir = current_dir / "暂存态"
ckpt_dir.mkdir(exist_ok=True)

# 2. 使用 PyTreeCheckpointer 而不是 StandardCheckpointer
checkpointer = ocp.PyTreeCheckpointer()

# 3. 保存模型（使用绝对路径）
# _, state = nnx.split(excited_model)
save_path = str(ckpt_dir / "excited_state1")  # 转为字符串
checkpointer.save(save_path, params_es, force=True)

print(f"模型已保存到: {save_path}")

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history['energy'],range(len(history['energy'])))


In [ ]:
final_loss, final_energy, final_std, final_grad, final_overlap = forces_expect_with_penalty(
    machine_es,
    params_gs,params_es,
    samples.reshape(-1,4), 1.0
)
final_error = jnp.abs(final_energy.real - E_fcis[1])
final_loss, final_energy, final_overlap

In [ ]:
final_energy = exact_energy_efficient(machine_es, params_es,hi,ha)
final_error = jnp.abs(final_energy.real - E_fcis[1])

print("\n" + "="*60)
print(f"训练完成！")
print(f"第一激发态能量：{final_energy:.8f}Ha")
print(f"FCI 激发态基准：{E_fcis[1]:.8f} Ha")
# print(f"正交重叠 |⟨Ψ0|Ψ1⟩|² = {final_overlap:.8f} (越小越好)")
print(f"绝对误差：{final_error:.6f} Ha")
print("="*60)